In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# Load the uploaded CSV file
df = pd.read_csv('/content/drive/MyDrive/spamham(email).csv')

In [3]:
# 1. Check Rows, Columns & Names
rows, cols = df.shape
print(f"Records (Rows): {rows}")
print(f"Columns: {cols}")
print(f"Column Names: {list(df.columns)}\n")

Records (Rows): 4845
Columns: 3
Column Names: ['Text', 'Class', 'Label']



In [4]:
# 2. Check Spam vs Ham Distribution
print("--- Spam / Ham Distribution ---")
print(df['Label'].value_counts())
print()

--- Spam / Ham Distribution ---
Label
ham     2732
spam    2113
Name: count, dtype: int64



In [5]:
# 3. Check Missing Values & Duplicates
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}\n")

Missing Values: 0
Duplicate Rows: 506



In [6]:
# 4. Display Example Messages
print("--- HAM EXAMPLES ---")
for i, msg in enumerate(df[df['Label'] == 'ham']['Text'].head(2), 1):
    print(f"Ham Example {i}:\n{msg[:200]}...\n")

print("--- SPAM EXAMPLES ---")
for i, msg in enumerate(df[df['Label'] == 'spam']['Text'].head(2), 1):
    print(f"Spam Example {i}:\n{msg[:200]}...\n")

--- HAM EXAMPLES ---
Ham Example 1:
over. SidLet me know. Thx....

Ham Example 2:
Not a surprising assessment from Embassy....

--- SPAM EXAMPLES ---
Spam Example 1:
Supply Quality China's EXCLUSIVE dimensions at Unbeatable Price.Dear Sir, We are pleased to inform you as one of China's largest export & import sto=ne company-Wanlistone Group, The Group its subsidia...

Spam Example 2:
Dear Friend,Greetings to you.I wish to accost you with a request that would be of immense benefit to both of us. Being an executor of wills, it is possible that we may be tempted to make fortune out o...



In [7]:
# 2. Extract and organize summary statistics into a dictionary
summary_data = {
    "Item": [
        "Records",
        "Columns",
        "Column Names",
        "Spam",
        "Ham",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Result": [
        len(df),                          # Total number of rows
        df.shape[1],                      # Total number of columns
        ", ".join(df.columns),            # Column names joined by commas
        (df['Label'] == 'spam').sum(),    # Total spam count
        (df['Label'] == 'ham').sum(),     # Total ham count
        df.isnull().sum().sum(),          # Total missing values across all cells
        df.duplicated().sum()             # Total duplicate rows
    ]
}

# 3. Convert the dictionary into a DataFrame and display as an HTML table
summary_df = pd.DataFrame(summary_data)
display(summary_df)

,Item,Result
0,Records,4845
1,Columns,3
2,Column Names,"Text, Class, Label"
3,Spam,2113
4,Ham,2732
5,Missing Values,0
6,Duplicate Rows,506


In [8]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [9]:
# 1. Download necessary NLTK datasets
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [10]:
# STEP 2: Load the Raw Email Dataset
print("\nLoading dataset...")
df = pd.read_csv('/content/drive/MyDrive/spamham(email).csv')
print(f"Dataset successfully loaded. Total records: {len(df)}")


Loading dataset...
Dataset successfully loaded. Total records: 4845


In [11]:
# Initialize Lemmatizer and Stop Words set
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [12]:
# STEP 3: Define Text Preprocessing Function
def preprocess_text(text):
    """
    Applies complete preprocessing pipeline on a single text string:
    1. Lowercasing
    2. URL tokenization ('urltoken')
    3. HTML tag removal
    4. Punctuation, special character & number removal
    5. Tokenization & Extra whitespace stripping
    6. Stop word removal
    7. Lemmatization
    """
    if not isinstance(text, str):
        return ""

    # 1. Convert text to lowercase
    text = text.lower()

    # 2. Replace web URLs with 'urltoken' placeholder
    text = re.sub(r'https?://?\S+|www\.\S+', ' urltoken ', text)

    # 3. Remove HTML tags (e.g., <p>, <br>, <html>)
    text = re.sub(r'<[^>]+>', ' ', text)

    # 4. Remove special characters, punctuation, and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # 5. Tokenize by whitespace (automatically handles extra spaces)
    words = text.split()

    # 6. Remove stop words & single-letter tokens, apply Lemmatization
    cleaned_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 1
    ]

    # 7. Rejoin tokens into a single clean string
    return " ".join(cleaned_words)

# ==========================================
# STEP 4: Run Preprocessing on Dataset
# ==========================================
print("\nApplying text preprocessing pipeline... Please wait.")
df['Cleaned_Text'] = df['Text'].apply(preprocess_text)
print("Preprocessing complete!")

# ==========================================
# STEP 5: Verify Results (Before vs After)
# ==========================================
print("\n" + "="*60)
print("PREPROCESSING VERIFICATION (BEFORE vs AFTER)")
print("="*60)

# Display Example 1: General Spam Email
print("\n--- Example 1: Standard Email ---")
print("BEFORE:")
print(df['Text'].iloc[0][:200] + "...\n")
print("AFTER:")
print(df['Cleaned_Text'].iloc[0][:200] + "...")

# Display Example 2: Email Containing a URL
url_rows = df[df['Text'].str.contains(r'http|www\.', case=False, na=False)]
if not url_rows.empty:
    sample_url_idx = url_rows.index[0]
    print(f"\n--- Example 2: Email with URL (Row Index {sample_url_idx}) ---")
    print("BEFORE:")
    print(df['Text'].iloc[sample_url_idx][:200] + "...\n")
    print("AFTER (Notice 'urltoken' replacing the link):")
    print(df['Cleaned_Text'].iloc[sample_url_idx][:200] + "...")

# ==========================================
# STEP 6: Export Cleaned Dataset
# ==========================================
# Select and rename columns for standard format
cleaned_df = df[['Cleaned_Text', 'Label', 'Class']].rename(
    columns={
        'Cleaned_Text': 'text',
        'Label': 'label',
        'Class': 'class'
    }
)

# Export cleaned dataframe to CSV
output_filename = '/content/drive/MyDrive/cleaned_email_dataset.csv'
cleaned_df.to_csv(output_filename, index=False)

print("\n" + "="*60)
print(f"SUCCESS: Cleaned dataset saved as '{output_filename}' ({len(cleaned_df)} rows).")
print("="*60)


Applying text preprocessing pipeline... Please wait.
Preprocessing complete!

PREPROCESSING VERIFICATION (BEFORE vs AFTER)

--- Example 1: Standard Email ---
BEFORE:
Supply Quality China's EXCLUSIVE dimensions at Unbeatable Price.Dear Sir, We are pleased to inform you as one of China's largest export & import sto=ne company-Wanlistone Group, The Group its subsidia...

AFTER:
supply quality china exclusive dimension unbeatable price dear sir pleased inform one china largest export import sto ne company wanlistone group group subsidiary specialize uarrying processing sale d...

--- Example 2: Email with URL (Row Index 2) ---
BEFORE:
Dear Friend,Greetings to you.I wish to accost you with a request that would be of immense benefit to both of us. Being an executor of wills, it is possible that we may be tempted to make fortune out o...

AFTER (Notice 'urltoken' replacing the link):
dear friend greeting wish accost request would immense benefit u executor will possible may tempted make fort

In [13]:
import pandas as pd

# Load the email dataset
df_email = pd.read_csv('/content/drive/MyDrive/cleaned_email_dataset.csv')

# Drop rows with missing text values
df_email = df_email.dropna(subset=['text']).reset_index(drop=True)
df_email = df_email[df_email['text'].str.strip() != ''].reset_index(drop=True)

# Save the updated email dataset
df_email.to_csv('/content/drive/MyDrive/updated_cleaned_email_dataset.csv', index=False)

print(f"✅ Success! Email dataset updated. Remaining clean records: {len(df_email)}")

✅ Success! Email dataset updated. Remaining clean records: 4770
